# Mamba-CL on Split-CIFAR100 (10 tasks x 10 classes): De-focus Mamba **base** backbone

Mamba-CL protocol from the official code: **10 tasks, 10 epochs/task**, null-space projection, seed 2024.
Backbone is `defocus_mamba_base` (24 layers, 768-dim, ~92M params, ImageNet-1K pretrained) instead of
`defocus_mamba_large` (48 layers, 1024-dim, ~330M, ImageNet-21K pretrained).
Our large-backbone reference run was 10 tasks x 10 classes: AA 90.16%, FM 2.18%.

Environment matches the RunPod setup: Python 3.11 venv, torch 2.3.1 + cu121, causal-conv1d 1.4.0, mamba-ssm 1.2.2.

**Settings:** GPU **T4 x2**, Internet on. One run uses both GPUs (`DATA_PARALLEL`): each batch of 200 is split 100 + 100. Attach these Kaggle Secrets (Add-ons > Secrets):
- `HF_TOKEN`: a Hugging Face token with **write** access. Checkpoints and `results.json` are mirrored to the private
  repo `HF_REPO`, so they survive the end of the session.
- `WANDB_API_KEY` (optional). Without it W&B logs offline; sync later with `wandb sync` on the saved `wandb/` folder.

**Resuming:** Kaggle stops sessions at 12 h. Start a new session and **Run All** with the same parameters. The run
pulls its checkpoint from `HF_REPO` and continues from the last finished epoch. A checkpoint is written after every
epoch (`epoch.pt`) and after every task (`task_end.pt`, `results.json`).

In [ ]:
# ---- run parameters ----
REPO     = "https://github.com/HasinthakaPiyumal/hippo-cortex.git"
BRANCH   = "feat/kaggle-progress-logging"
MODEL    = "defocus_mamba_base"   # or "vim_small" (Vim-small, hustvl ImageNet weights; the Inf-SSM backbone)
TASKS    = 10        # Split-CIFAR100 10x10, as in Mamba-CL and Inf-SSM
EPOCHS   = 10        # Mamba-CL official code default (the paper text says 50)
BATCH    = 200       # Mamba-CL protocol; with DATA_PARALLEL each T4 gets 100. Single T4: use 128 (200 ran out of memory)
SEED     = 2024
USE_AMP  = True
DATA_PARALLEL = True    # split each batch across both T4s (one run); False = single GPU
WORKERS  = 4
WANDB_PROJECT = "mamba-cl-baseline"
PROGRESS_INTERVAL = 30   # seconds between progress-bar updates in the log

# ---- checkpoints ----
CKPT_DIR = "/kaggle/working/checkpoints"
HF_REPO  = "<hf-username>/hippocortex-checkpoints"   # private model repo, created on first run
HF_SYNC_EPOCHS = True    # upload epoch.pt after every epoch, not only after each task
HF_SQUASH = True         # squash HF_REPO's history after each task, so it keeps one copy per file (use a dedicated repo)
assert "<" not in HF_REPO, "set HF_REPO to your Hugging Face username/repo"

In [ ]:
!nvidia-smi
import os, subprocess
os.chdir("/kaggle/working")
if not os.path.exists("hippo-cortex"):
    # Fall back to main once the branch is merged (GitHub deletes merged branches here).
    if subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO]).returncode != 0:
        subprocess.run(["git", "clone", "--depth", "1", REPO], check=True)
ROOT = "/kaggle/working/hippo-cortex"
VENV = "/kaggle/working/py311_env"
PY, PIP = f"{VENV}/bin/python", f"{VENV}/bin/pip"
!cd {ROOT} && git log --oneline -1

In [ ]:
# Python 3.11 venv (the Kaggle image's Python is too new for torch 2.3.1 / mamba-ssm 1.2.2)
!add-apt-repository -y ppa:deadsnakes/ppa > /dev/null
!apt-get update -y -qq > /dev/null
!apt-get install -y -qq python3.11 python3.11-venv python3.11-dev python3.11-distutils > /dev/null
!python3.11 -m venv {VENV}
!{PY} -m ensurepip
!{PIP} install -q --upgrade pip
!{PIP} install -q torch==2.3.1 torchvision==0.18.1 torchaudio==2.3.1 --index-url https://download.pytorch.org/whl/cu121
!{PIP} install -q ninja packaging wheel
!{PIP} install -q --no-build-isolation causal-conv1d==1.4.0
!{PIP} install -q --no-build-isolation mamba-ssm==1.2.2
!cd {ROOT} && {PIP} install -q -r requirements.txt && {PIP} install -q -U wandb && {PIP} install -q huggingface_hub && {PIP} install -q --no-deps -e .
!{PY} -c "import torch, mamba_ssm, causal_conv1d; from mamba_ssm.ops.selective_scan_interface import selective_scan_fn; print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), '| mamba fast path OK')"

In [ ]:
if MODEL == "vim_small":
    CKPT = "vim_small_in1k"   # train_eval.py downloads the hustvl Vim-small ImageNet weights from HF
else:
    CKPT = "/kaggle/working/defocus_mamba_base_cls.pth"
    if not os.path.exists(CKPT):
        !wget -q -O {CKPT} https://github.com/OpenGVLab/De-focus-Attention-Networks/releases/download/v1.0/defocus_mamba_base_cls.pth
    !ls -la {CKPT}
DATA = "/tmp/cifar100-images"
if not os.path.exists(f"{DATA}/val"):
    !cd {ROOT} && {PY} scripts/download_datasets.py --dataset cifar100 && {PY} scripts/split_cifar100.py && cp -r data/cifar100-images /tmp/
!ls {DATA}/train | wc -l; ls {DATA}/val | wc -l

In [ ]:
# Tokens come from Kaggle Secrets; they are never written into this notebook.
env = dict(os.environ, WANDB_PROJECT=WANDB_PROJECT, WANDB_DIR="/kaggle/working")
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
env["HF_TOKEN"] = secrets.get_secret("HF_TOKEN").strip()   # required: checkpoints are mirrored to HF_REPO
# Check the token here, before training starts, rather than failing inside train_eval.py.
from huggingface_hub import HfApi
me = HfApi(token=env["HF_TOKEN"]).whoami()   # 401 here = the HF_TOKEN secret is wrong, expired or revoked
spaces = [me["name"]] + [o["name"] for o in me.get("orgs", [])]
assert HF_REPO.split("/")[0] in spaces, f"HF_REPO namespace must be one of {spaces}"
print(f"HF: logged in as {me['name']}, mirroring checkpoints to {HF_REPO}")
try:
    env["WANDB_API_KEY"] = secrets.get_secret("WANDB_API_KEY")
    print("W&B: online (key from Kaggle Secret)")
except Exception as e:
    env["WANDB_MODE"] = "offline"
    print("W&B: secret WANDB_API_KEY not attached, logging offline:", type(e).__name__)

In [ ]:
import time
RUN_NAME = f"mamba_cl_{MODEL}_cifar100_t{TASKS}_e{EPOCHS}_seed_{SEED}"   # same name train_eval.py uses
LOG = f"/kaggle/working/{RUN_NAME}_{time.strftime('%Y%m%d-%H%M%S')}.log"   # one log per session
cmd = [PY, "-u", "train_eval.py", "-d", "cifar100", "-m", MODEL,
       "-t", str(TASKS), "-e", str(EPOCHS), "-b", str(BATCH),
       "--pretrained_path", CKPT, "--data_root", DATA,
       "--null_eta", "0.95", "--use_null_space", "--seed", str(SEED),
       "--use_amp", str(USE_AMP), "--data_parallel", str(DATA_PARALLEL), "-jt", str(WORKERS), "--use_wandb",
       "--progress_interval", str(PROGRESS_INTERVAL),
       "--ckpt_dir", CKPT_DIR, "--hf_repo", HF_REPO,
       "--hf_sync_epochs", str(HF_SYNC_EPOCHS), "--hf_squash", str(HF_SQUASH)]
print(" ".join(cmd))
t0 = time.time(); done = 0
with open(LOG, "w") as f:
    p = subprocess.Popen(cmd, cwd=f"{ROOT}/baselines/mamba-cl", env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        f.write(line); f.flush()
        print(line, end="", flush=True)   # full log: epochs, progress bars, null-space timing, checkpoints
        if "Results of task" in line:
            done += 1
            el = time.time() - t0
            print(f"   this session: {done} task(s) in {el/3600:.2f} h, {el/done/3600:.2f} h per task", flush=True)
    p.wait()
print("exit code", p.returncode, f"in {(time.time()-t0)/3600:.2f} h; full log: {LOG}")

In [ ]:
# Keep this session's log next to the checkpoints on HF, then show the results so far.
import json
from huggingface_hub import HfApi
try:
    HfApi(token=env["HF_TOKEN"]).upload_file(path_or_fileobj=LOG, path_in_repo=f"{RUN_NAME}/logs/{os.path.basename(LOG)}",
                                             repo_id=HF_REPO, commit_message="session log")
    print("log uploaded to", f"{HF_REPO}/{RUN_NAME}/logs/")
except Exception as e:
    print("log upload failed:", e)

res_path = f"{CKPT_DIR}/{RUN_NAME}/results.json"
if os.path.exists(res_path):
    r = json.load(open(res_path))
    print(f"tasks done {r['tasks_done']}/{r['num_tasks']} | AA {r['avg_acc']:.2%} | FM {r['forgetting']:.2%} | {r['elapsed_hours']:.2f} h")
    print("AA per task:", " ".join(f"{a:.2%}" for a in r["avg_acc_per_task"]))
else:
    print("no results.json yet")

In [ ]:
!tail -n 40 {LOG}